In [107]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
import pandas as pd
import numpy as np
import json
import random
import ipywidgets as widgets
from src.utils import df_features_to_numpy, df_target_to_numpy, run_experiment_pytorch_clasificacion, run_experiment_pytorch_regresion, permutation_importance_pytorch
from sklearn.preprocessing import StandardScaler
from src.splitting import split_estratificado_autor
from sklearn.neighbors import NearestNeighbors
from IPython.display import display

# Cargo los datasets

In [2]:
#LEO TEST
chunks_test = pd.read_json("set_test_1.json", lines=True, chunksize=100_000)
chunk_list_test = []
for chunk in chunks_test:
    chunk_list_test.append(chunk)
df_test = pd.concat(chunk_list_test, ignore_index=True)

#LEO VAL
chunks_val = pd.read_json("set_val_1.json", lines=True, chunksize=100_000)
chunk_list_val = []
for chunk in chunks_val:
    chunk_list_val.append(chunk)
df_val = pd.concat(chunk_list_val, ignore_index=True)

#LEO TRAIN
chunks_train = pd.read_json("set_train_1.json", lines=True, chunksize=100_000)
chunk_list_train = []
for chunk in chunks_train:
    chunk_list_train.append(chunk)
df_train = pd.concat(chunk_list_train, ignore_index=True)

#LEO AUTORES
chunks_autores = pd.read_json("goodreads_book_authors.json", lines=True, chunksize=100_000)
chunk_list_autores = []
for chunk in chunks_autores:
    chunk_list_autores.append(chunk)
df_autores = pd.concat(chunk_list_autores, ignore_index=True)

# Preparacion de los datos

Utilizo el dataset de autores para asociar los IDs que ya tenemos, con sus nombres correspondientes:

In [3]:
descripcion_emb_train = np.load("descripcion_emb_train.npy")
descripcion_emb_val = np.load("descripcion_emb_val.npy")
descripcion_emb_test = np.load("descripcion_emb_test.npy")

In [4]:
#Creo las columnas para el dataset:
descripcion_colum = [f"description_emb_{num}" for num in range(50)]

descripcion_emb_train_df = pd.DataFrame(descripcion_emb_train, columns = descripcion_colum, index = df_train.index)
df_train = df_train.join(descripcion_emb_train_df)

descripcion_emb_val_df = pd.DataFrame(descripcion_emb_val, columns = descripcion_colum, index = df_val.index)
df_val = df_val.join(descripcion_emb_val_df)

descripcion_emb_test_df = pd.DataFrame(descripcion_emb_test, columns = descripcion_colum, index = df_test.index)
df_test = df_test.join(descripcion_emb_test_df)

In [5]:
#UNIFICO TRAIN, VAL, TEST EN UN UNICO DF 
df = pd.concat([df_train, df_val, df_test], ignore_index=True)

In [6]:
#Al hacer merge, usamos "how = left" para que quedarnos con todas las muestras de df_train (aunque no lo encuentre coincidencia).
#Si no encuentra el ID de df en df_autores, simplemente completa con NaN
df = df.merge(df_autores[["author_id", "name"]], left_on = "author_id_principal", right_on = "author_id", how = "left")
#Despues del merge, tenemos entonces entre las columnas a : "author_id_principal", "author_id" y "author_name"
#Pero como son el mismo ID, elimino una de las dos:
df = df.drop(columns = "author_id")
#Renombro la columna:
df = df.rename(columns = {"name": "author_name"})
df["titulo_autor"] = (df["title"] + " — " + df["author_name"]) #Para la interfaz

Es mas facil eliminar las features con las que no queremos entrenar, que quedarnos con las que si, son muchas mas. 

In [7]:
df_recomendador = df[["titulo_autor", "title", "author_name"]]
df = df.drop(columns = ["title_key", "publisher", "title_text", "author_name", "titulo_autor", "language_code", "ratings_count", "authors", "format_group", "genres", "description", "title"])

Dejo esto por si queremos testear solo con las features basicas

In [115]:
#generos = ["fiction", "non fiction", "fantasy", "science fiction", "mystery", "thriller", "romance", "horror", "historical", "young adult", "children", "adventure", "contemporary", "dystopian", "paranormal", "urban", "crime", "biography", "memoir", "self help", "history", "psychology", "poetry", "classics", "graphic novel", "short stories", "humor", "religion"]
#features = ["is_series", "publication_year", "num_pages", 'AUDIO', 'COMIC', 'EBOOK', 'HARDCOVER', 'OTHER', 'PAPERBACK', "language_code_ang", "language_code_en", "language_code_en-CA", "language_code_en-GB", "language_code_en-US", "language_code_eng", "language_code_enm", "language_code_gem", "language_code_sco", "cantidad_generos"] + generos #'format_group_UNKNOWN' la saco por el one hot en regresion
#
##Me quedo solo con las features necesarias para el modelo base y las estandarizo
#dfs = [df_train, df_val, df_test]
#y_train_regresion, y_val_regresion, y_test_regresion = df_target_to_numpy("average_rating", dfs)
#y_train_clasificacion, y_val_clasificacion, y_test_clasificacion = df_target_to_numpy("estrellas", dfs)
#X_train, X_val, X_test = df_features_to_numpy(features, dfs)
#
##estandarizo los datos
#scaler = StandardScaler()
#X_train_scaled = scaler.fit_transform(X_train)
#X_val_scaled = scaler.transform(X_val) #escala X_val con los valores sacados de X_train
##TODO: escalar el X_test

In [8]:
X = df.to_numpy()

#estandarizo los datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Implementacion del modelo

In [9]:
modelo = NearestNeighbors(n_neighbors = 20, metric = "cosine")
modelo.fit(X_scaled)

,n_neighbors,20
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


# Desarrollando recomendador

Una vez ya entrenado el modelo, vuelvo a unificar df_recomendador (con las features no numericas necesarias para identificar al libro) y df (con las features necesarias para entrenar)

In [10]:
df = pd.concat([df_recomendador, df], axis=1)

In [11]:
def recomendar(modelo, X, indice_libro, n_recomendaciones = 10):
    distancias, indices = modelo.kneighbors(X[indice_libro].reshape(1, -1), n_neighbors = n_recomendaciones + 1)
    return indices[0][1:], distancias[0][1:]

Ejemplo de recomendador para un libro especifico:

In [121]:
idx = df[df_recomendador["title"] == "The Book Thief"].index[0]
recomendados, distancias = recomendar(modelo, X_scaled, idx, 10)

print("\nRecomendaciones:")
for idx, dist in zip(recomendados, distancias):
    print(f"{df_recomendador.iloc[idx]['title']}  (distancia = {dist:.4f})")


Recomendaciones:
The Book of Aron  (distancia = 0.1868)
Alone In Berlin  (distancia = 0.2710)
The Color Purple  (distancia = 0.2764)
The Reader  (distancia = 0.2880)
Revolution  (distancia = 0.2998)
The Orphan's Tale  (distancia = 0.3103)
The Idiot  (distancia = 0.3125)
The Secret Life of Bees  (distancia = 0.3174)
Salvage the Bones  (distancia = 0.3175)
Americanah  (distancia = 0.3236)


### Trato de implementar lo de Titulo + autor

In [18]:
print(df.columns.tolist())

['titulo_autor', 'title', 'author_name', 'author_id_principal', 'publication_year', 'num_pages', 'average_rating', 'is_series', 'AUDIO', 'COMIC', 'EBOOK', 'HARDCOVER', 'OTHER', 'PAPERBACK', 'UNKNOWN', 'adventure', 'biography', 'children', 'classics', 'contemporary', 'crime', 'dystopian', 'fantasy', 'fiction', 'graphic novel', 'historical', 'history', 'horror', 'humor', 'memoir', 'mystery', 'non fiction', 'paranormal', 'poetry', 'psychology', 'religion', 'romance', 'science fiction', 'self help', 'short stories', 'thriller', 'urban', 'young adult', 'language_code_ang', 'language_code_en', 'language_code_en-CA', 'language_code_en-GB', 'language_code_en-US', 'language_code_eng', 'language_code_enm', 'language_code_gem', 'language_code_sco', 'cantidad_generos', 'estrellas', 'description_emb_0', 'description_emb_1', 'description_emb_2', 'description_emb_3', 'description_emb_4', 'description_emb_5', 'description_emb_6', 'description_emb_7', 'description_emb_8', 'description_emb_9', 'descript

In [12]:
#Para evitar tener que recorrer el dataset de libros cada vez que quiero buscar un libro, 
#me creo un diccionario que asocia "titulo_autor" con su indice:
titulo_a_indice = dict(zip(df["titulo_autor"], df.index)) #Reduce el tiempo de busqueda, lo que nos sirve con el dataset grande

In [ ]:
# def recomendar_por_libro(modelo, libros_df, X, titulo_autor, titulo_a_indice = titulo_a_indice, cant_recomendaciones = 5):
#     #Observaciones: la interfaz ya obliga al usuario a elegir un unico libro (al especificar tambien el autor),
#     #entonces ya no hace chequear el caso en el que len(resultados) > 1

#     #Ahora hay que ver que hacemos con cada posible situacion:
#     #Primer caso: No encontramos ningun libro con esos datos
#     if (titulo_autor not in titulo_a_indice):
#         print("No se encontro ningun libro con esos datos.")
#         return None
    
#     #Segundo caso: se encontro el libro
#     indice_libro = titulo_a_indice[titulo_autor]
#     #Obtengo las recomendaciones:
#     libros_recomendados, distancias = recomendar(modelo, X, indice_libro, cant_recomendaciones)
#     recomendaciones = libros_df.iloc[libros_recomendados].copy()
#     recomendaciones["distancia"] = distancias
#     columnas = ["title", "author_name", "publication_year"]
#     return recomendaciones[columnas]

In [14]:
all_titles = sorted(df["titulo_autor"].dropna().unique().tolist())

book_selector = widgets.Combobox(
    placeholder="Escribí un libro...",
    options=[],
    description="Libro:",
    ensure_option=False
)

def actualizar_opciones(change):
    texto = change["new"] or ""
    if len(texto) < 2:
        book_selector.options = []
        return
    matches = [t for t in all_titles if texto.lower() in t.lower()][:10]
    book_selector.options = matches

book_selector.observe(actualizar_opciones, names="value")

boton_recomendar = widgets.Button(
    description="Recomendar",
    button_style="success",
    icon="search"
)

salida = widgets.Output()

def al_hacer_click(boton):
    with salida:
        salida.clear_output()
        titulo_autor = book_selector.value
        recomendaciones = recomendar_por_libro(
            modelo,
            df,
            X_scaled,
            titulo_autor,
            titulo_a_indice
        )
        if recomendaciones is not None:
            display(recomendaciones)

boton_recomendar.on_click(al_hacer_click)

display(book_selector)
display(boton_recomendar)
display(salida)

Combobox(value='', description='Libro:', placeholder='Escribí un libro...')

Button(button_style='success', description='Recomendar', icon='search', style=ButtonStyle())

Output()

In [15]:
all_titles = sorted(df["titulo_autor"].dropna().unique().tolist())

search_text = widgets.Text(
    placeholder="Escribí un libro...",
    description="Buscar:"
)

match_selector = widgets.Select(
    options=[],
    rows=10,
    layout=widgets.Layout(height="220px", width="600px")
)

boton_recomendar = widgets.Button(
    description="Recomendar",
    button_style="success",
    icon="search"
)

salida = widgets.Output()

def actualizar_opciones(change):
    texto = (change["new"] or "").strip().lower()
    if texto == "":
        match_selector.options = []
        return
    matches = [t for t in all_titles if texto in t.lower()][:50]
    match_selector.options = matches

def al_hacer_click(boton):
    with salida:
        salida.clear_output()
        titulo_autor = match_selector.value
        recomendaciones = recomendar_por_libro(
            modelo,
            df,
            X_scaled,
            titulo_autor,
            titulo_a_indice
        )
        if recomendaciones is not None:
            display(recomendaciones)

search_text.observe(actualizar_opciones, names="value")
boton_recomendar.on_click(al_hacer_click)

display(search_text)
display(match_selector)
display(boton_recomendar)
display(salida)

Text(value='', description='Buscar:', placeholder='Escribí un libro...')

Select(layout=Layout(height='220px', width='600px'), options=(), rows=10, value=None)

Button(button_style='success', description='Recomendar', icon='search', style=ButtonStyle())

Output()

### Trato ahora una idea de interfaz

In [16]:
def recomendar_por_libro(modelo, libros_df, X, titulo_autor, titulo_a_indice = titulo_a_indice, cant_recomendaciones = 5):
    #Observaciones: la interfaz ya obliga al usuario a elegir un unico libro (al especificar tambien el autor),
    #entonces ya no hace chequear el caso en el que len(resultados) > 1

    #Ahora hay que ver que hacemos con cada posible situacion:
    #Primer caso: No encontramos ningun libro con esos datos
    if (titulo_autor not in titulo_a_indice):
        print("No se encontro ningun libro con esos datos.")
        return None
    
    #Segundo caso: se encontro el libro
    indice_libro = titulo_a_indice[titulo_autor]
    #Obtengo las recomendaciones:
    libros_recomendados, distancias = recomendar(modelo, X, indice_libro, cant_recomendaciones)
    recomendaciones = libros_df.iloc[libros_recomendados].copy()
    recomendaciones["distancia"] = distancias
    recomendaciones["similitud"] = (1 - recomendaciones["distancia"]) * 100 #Usamos esta formula porque estamos usando la metrica "cosine"
    columnas = ["title", "author_name", "publication_year", "average_rating", "similitud"]
    return recomendaciones[columnas]

In [17]:
from IPython.display import HTML, display

def mostrar_recomendaciones(recomendaciones):

    html = """
    <h2 style="
        color:#2E86C1;
        font-family:Arial;
        margin-bottom:20px;
    ">
        📚 Libros recomendados
    </h2>

    <div style="
        display:flex;
        flex-wrap:wrap;
        gap:20px;
        justify-content:flex-start;
        align-items:flex-start;
    ">
    """

    for _, libro in recomendaciones.iterrows():

        html += f"""
        <div style="
            width:270px;
            min-height:280px;
            background:#F8F9FA;
            border:1px solid #D6DBDF;
            border-radius:15px;
            padding:18px;
            box-shadow:3px 3px 12px rgba(0,0,0,0.12);
            display:flex;
            flex-direction:column;
            justify-content:space-between;
            transition:0.2s;
        ">

            <div>

                <h3 style="
                    margin-top:0;
                    height:60px;
                    overflow:hidden;
                    color:#2C3E50;
                    font-family:Arial;
                ">
                    📖 {libro["title"]}
                </h3>

                <hr style="border:0;border-top:1px solid #E5E7E9;">

                <p style="
                    font-size:15px;
                    line-height:1.6;
                    color:#34495E;
                    font-family:Arial;
                ">

                👤 <b>Autor</b><br>
                {libro["author_name"]}<br><br>

                📅 <b>Año</b><br>
                {int(libro["publication_year"])}

                </p>

            </div>

            <div>

                <span style="
                    background:#FFF3CD;
                    color:#856404;
                    padding:6px 10px;
                    border-radius:8px;
                    font-weight:bold;
                    font-size:14px;
                ">
                    ⭐ {libro["average_rating"]:.2f}
                </span>

                <span style="
                    background:#D4EDDA;
                    color:#155724;
                    padding:6px 10px;
                    border-radius:8px;
                    font-weight:bold;
                    font-size:14px;
                    margin-left:8px;
                ">
                    🎯 {libro["similitud"]:.1f}%
                </span>

            </div>

        </div>
        """

    html += "</div>"

    display(HTML(html))

In [19]:
all_titles = sorted(df["titulo_autor"].dropna().unique().tolist())

book_selector = widgets.Combobox(
    placeholder="Escribí un libro...",
    options=[],
    description="Libro:",
    ensure_option=False,
    layout=widgets.Layout(width="700px")
)

def actualizar_opciones(change):
    texto = change["new"] or ""
    if len(texto) < 2:
        book_selector.options = []
        return
    matches = [t for t in all_titles if texto.lower() in t.lower()][:10]
    book_selector.options = matches

book_selector.observe(actualizar_opciones, names="value")

boton_recomendar = widgets.Button(
    description="Recomendar",
    button_style="success",
    icon="search",
    layout=widgets.Layout(width="220px", height="45px")
)

salida = widgets.Output()

def al_hacer_click(boton):
    with salida:
        salida.clear_output()
        titulo_autor = book_selector.value
        recomendaciones = recomendar_por_libro(
            modelo,
            df,
            X_scaled,
            titulo_autor,
            titulo_a_indice
        )
        if recomendaciones is not None:
            mostrar_recomendaciones(recomendaciones)

boton_recomendar.on_click(al_hacer_click)

titulo = widgets.HTML(
"""
<h1 style="color:#2E86C1;">
📚 Recomendador de Libros
</h1>

<p>
Escribí el nombre de un libro y elegilo de la lista.
</p>
"""
)

interfaz = widgets.VBox([
    titulo,
    book_selector,
    boton_recomendar,
    salida
])

display(interfaz)